# 15 - The Audit Evidence Packet

"Advisory" does not mean unaccountable. Every Metatate answer carries a
`decision_id`, publication provenance, and cited policy versions — this
notebook turns a day of governed questions into a single audit-ready
report: decisions with receipts, the `explain_why` chain proving each
one is still CURRENT, and the honest corners where the estate refused
to guess. The reusable `audit_evidence` package does the assembly; the
server keeps its own corroborating ledger (MCP Tools → Tokens →
**View requests**).


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
if mode == "live" and not os.getenv("METATATE_MCP_URL"):
    print("Live mode needs a Metatate endpoint. Fastest path (about 5 minutes):")
    print("  1. Create a free account: https://app.getmetatate.com/sign-up?ref=examples")
    print("  2. Workspace dashboard: 'Load the demo' banner -> 'Load the AcmeCloud demo'")
    print("  3. MCP Tools -> Tokens: issue a token; Connect tab has your endpoint URL")
    print("  4. export METATATE_MCP_URL=... METATATE_SAAS_MCP_TOKEN=...")
    print("     (full steps: docs/live-mode-saas.md)")

client = get_client()
print(f"Metatate examples mode: {mode}")


def asset(table, column=None, schema="public"):
    ref = {"database": "acmecloud_demo", "schema": schema, "table": table}
    if column:
        ref["column"] = column
    return ref


def answer_label(answer):
    state = answer.get("state")
    if state and state != "answered":
        return state
    return answer.get("decision") or answer.get("verdict") or "unknown"


def print_answer(answer):
    print(f"state:    {answer.get('state')}")
    if "decision" in answer:
        print(f"decision: {answer['decision']}")
    if "verdict" in answer:
        print(f"verdict:  {answer['verdict']}")
    if answer.get("reason"):
        print(f"reason:   {answer['reason']}")
    for condition in answer.get("conditions") or []:
        print(f"condition [{condition.get('kind')}]: {condition.get('requirement')}")
    for prohibition in answer.get("prohibitions") or []:
        print(f"prohibition: {prohibition.get('detail')}")
    for obligation in answer.get("obligations") or []:
        print(f"obligation [{obligation.get('type')}]: {obligation.get('target')}")
    if "can_proceed_now" in answer:
        print(f"can_proceed_now: {answer['can_proceed_now']}")


## A day of governed questions


In [ ]:
from audit_evidence import collect_evidence, render_markdown

packet = collect_evidence(client)
print(f"decisions: {packet.total}")
print(f"explained and current: {packet.current}/{packet.explained}")
print(f"honest corners: {packet.honest_corners}")
print(f"publication: {packet.publication_id}")


## The packet, audit-ready

Per decision: asset, scenario, typed decision, the citing policy BY
NAME AND VERSION, evidence id, conditions and obligations — and the
explain-chain confirmation. Then the corners: the ungoverned legacy
table and the monitored custom mask, on the record instead of hidden.


In [ ]:
print(render_markdown(packet))


## Codify YOUR questions

`collect_evidence(client, questions=...)` takes any list shaped like
`DEFAULT_QUESTIONS` — codify the data-use questions your team answers
every quarter and regenerate this packet on a schedule. After a
republish, superseded decisions explain with `current: false` —
historical, honestly labeled, never rewritten (walkthrough:
publish-flip).
